# SPHERE Edge IDS/IPS Lab

A reproducible SPHERE experiment.

**Testbed:** SPHERE / MergeTB · **Project:** `fabricsphere` · **Experiment:** `edgeids` · **Realization:** `ids`

> Run this **inside the experiment's XDC JupyterLab** (the XDC must be attached to this realization). Execute the cells top-to-bottom.

## What this experiment does

This is an **edge intrusion-detection / prevention lab**.

- A **compromised** edge device (`edge`) runs a realistic **multi-stage attack** against the gateway: reconnaissance (SYN/FIN/XMAS port scans) → **web exploitation** (path traversal, SQL-injection, malicious user-agent against the gateway's web service) → **SSH brute force** → **DNS-tunneling exfiltration**.
- A **benign** edge device (`edge2`) generates only normal traffic (web + ping), giving you a healthy baseline to contrast against.
- The **gateway** (`gw`) hosts the victim web service + a DNS resolver and **mirrors** all edge-LAN traffic (software SPAN) to the sensor.
- The **sensor** (`mon`) runs **Suricata** with categorized rules and logs alerts to `eve.json`.
- Finally a **closed-loop IPS** identifies the attacker from the alerts and auto-blocks it on the gateway with `nftables` — and you verify the benign device keeps working.

### Data flow
```
 edge  (attacker) ─┐
                   ├─edge_lan─▶ gw ─(tc mirror)─▶ mon (Suricata IDS)
 edge2 (benign)  ──┘            │  victim web + DNS    │ categorized alerts
                                └────── IPS auto-block ◀┘ (nft drop attacker)
```

### Topology
`edge` ── `edge2`  
`gw` ── `mon`

### Nodes
| Node | Facility | Addresses | Setup script |
|------|----------|-----------|--------------|
| `gw` | moddeter | 10.0.1.1/24, 10.0.2.1/24 | `gw-setup.sh` |
| `edge` | moddeter | 10.0.1.10/24 | `edge-setup.sh` |
| `edge2` | moddeter | 10.0.1.11/24 | `edge2-setup.sh` |
| `mon` | moddeter | 10.0.2.10/24 | `mon-setup.sh` |

### What each cell does
1. **Connect** — SSH helpers + reachability for every node.
2. **Deploy** — push & run each setup script on its target node.
3. **Verify** — firewall / mirror / IDS / attacker & benign generators.
4. **Analyze** — let the attack run, then collect Suricata alerts + flows.
5. **Visualize** — topology, alert categories, severity, top talkers, attack graph, timeline, benign-vs-compromised.
6. **Respond (IPS)** — auto-block the attacker on the gateway and confirm the benign device still works.

## 1. Connect — SSH helpers + reachability

In [ ]:
import subprocess, os, json, time

EXPERIMENT = "edgeids"
REALIZATION = "ids"
PROJECT = "fabricsphere"
NODES = ["gw", "edge", "edge2", "mon"]
SCRIPTS = {"gw-setup.sh": "gw", "mon-setup.sh": "mon", "edge-setup.sh": "edge", "edge2-setup.sh": "edge2"}  # filename -> target node ('all' = every node)
NODE_ADDRS = {"gw": ["10.0.1.1/24", "10.0.2.1/24"], "edge": ["10.0.1.10/24"], "edge2": ["10.0.1.11/24"], "mon": ["10.0.2.10/24"]}   # node -> [cidr,...]
IP_TO_NODE = {"10.0.1.1": "gw", "10.0.2.1": "gw", "10.0.1.10": "edge", "10.0.1.11": "edge2", "10.0.2.10": "mon"}   # bare ip -> node
GATEWAY = "gw"         # node that forwards/mirrors (IPS enforcement point)
# Full management DNS as a fallback if bare node names don't resolve here.
NODE_DNS = {n: f'{n}.infra.{REALIZATION}.{EXPERIMENT}.{PROJECT}' for n in NODES}
SSH_OPTS = ['-o', 'StrictHostKeyChecking=no', '-o', 'BatchMode=yes', '-o', 'ConnectTimeout=10']
_host_cache = {}

def _host(node):
    '''Pick a reachable ssh host for *node*: bare name first, then full DNS.'''
    if node in _host_cache:
        return _host_cache[node]
    for cand in (node, NODE_DNS.get(node, node)):
        r = subprocess.run(['ssh', *SSH_OPTS, cand, 'true'], capture_output=True, text=True)
        if r.returncode == 0:
            _host_cache[node] = cand; return cand
    _host_cache[node] = node; return node  # let the real command surface the error

def sh(node, cmd, timeout=900):
    return subprocess.run(['ssh', *SSH_OPTS, _host(node), cmd],
                          capture_output=True, text=True, timeout=timeout)

def push(node, local, remote='~/'):
    return subprocess.run(['scp', *SSH_OPTS, '-q', local, f'{_host(node)}:{remote}'],
                          capture_output=True, text=True, timeout=300)

print('Checking connectivity...')
for n in NODES:
    r = sh(n, 'hostname; ip -4 -br addr')
    print(f'== {n}  (ssh {_host(n)})  rc={r.returncode} ==')
    print((r.stdout or r.stderr).rstrip())


## 2. Deploy — push & run the setup scripts

In [ ]:
# Push each bundled setup script to its target node(s) and run it with sudo.
# Installs packages (apt) — the first run can take a few minutes per node.
for script, target in SCRIPTS.items():
    path = os.path.join('scripts', script)
    if not os.path.exists(path):
        print(f'! missing {path}'); continue
    dests = NODES if target in ('all', '', None) else [target]
    for node in dests:
        up = push(node, path)
        if up.returncode != 0:
            print(f'! scp {script} -> {node} failed: {up.stderr.strip()}'); continue
        r = sh(node, f'chmod +x ~/{script} && sudo ~/{script}')
        print(f'\n===== {node}: {script}  (rc={r.returncode}) =====')
        print((r.stdout or '')[-3000:])
        if r.returncode != 0:
            print('STDERR:', (r.stderr or '')[-1500:])


## 3. Verify — tooling is running

In [ ]:
# Confirm the tooling is running on each node (firewall+mirror+victim on gw,
# Suricata on mon, attacker generator on edge, benign generator on edge2).
CHECKS = {
    'gw':    'echo ip_forward=$(cat /proc/sys/net/ipv4/ip_forward); '
             'echo victim_web=$(systemctl is-active victim-web 2>/dev/null); '
             'echo --- mirror ---; sudo tc -s qdisc show 2>/dev/null | grep -A1 ingress || echo no-mirror',
    'mon':   'echo suricata=$(systemctl is-active suricata 2>/dev/null); '
             'ls -l /var/log/suricata/ 2>/dev/null | head',
    'edge':  'echo attacker=$(systemctl is-active edge-traffic 2>/dev/null); ip route | grep -v default',
    'edge2': 'echo benign=$(systemctl is-active edge2-traffic 2>/dev/null)',
}
for node in NODES:
    cmd = CHECKS.get(node, 'systemctl --type=service --state=running --no-pager 2>/dev/null | head')
    r = sh(node, cmd)
    print(f'== {node} ==')
    print((r.stdout or r.stderr).rstrip(), '\n')


## 4. Analyze — run the attack & collect IDS alerts + flows

In [ ]:
# Let the attack run, then collect Suricata alerts + flow records from
# eve.json into structures the visualizations and IPS cell use.
SENSOR = 'mon' if 'mon' in NODES else (NODES[-1] if NODES else None)
WAIT_S = 90
ALERTS = []   # [{ts, signature, category, severity, src, dest}]
FLOWS = []    # [{proto, bytes, src, dest}]
counts = {}   # signature -> count
by_src = {}   # src ip -> alert count
by_cat = {}   # category -> count
by_sev = {}   # severity -> count
if not SENSOR:
    print('No sensor node.')
else:
    print(f'Letting the multi-stage attack run for {WAIT_S}s...')
    time.sleep(WAIT_S)
    raw = sh(SENSOR, 'sudo tail -c 4000000 /var/log/suricata/eve.json 2>/dev/null').stdout
    for line in raw.splitlines():
        try:
            ev = json.loads(line)
        except Exception:
            continue
        et = ev.get('event_type')
        if et == 'alert':
            a = ev.get('alert', {})
            sig = a.get('signature', '(unknown)')
            cat = a.get('category', 'uncategorized')
            sev = a.get('severity', 3)
            src = ev.get('src_ip', '')
            ALERTS.append({'ts': ev.get('timestamp', ''), 'signature': sig,
                           'category': cat, 'severity': sev,
                           'src': src, 'dest': ev.get('dest_ip', '')})
            counts[sig] = counts.get(sig, 0) + 1
            by_src[src] = by_src.get(src, 0) + 1
            by_cat[cat] = by_cat.get(cat, 0) + 1
            by_sev[sev] = by_sev.get(sev, 0) + 1
        elif et == 'flow':
            f = ev.get('flow', {})
            FLOWS.append({'proto': ev.get('proto', '?'),
                          'bytes': (f.get('bytes_toserver', 0) + f.get('bytes_toclient', 0)),
                          'src': ev.get('src_ip', ''), 'dest': ev.get('dest_ip', '')})
    print(f'\nCollected {len(ALERTS)} alert(s) / {len(FLOWS)} flow(s).')
    print('By category:', dict(sorted(by_cat.items(), key=lambda x: -x[1])))
    print('By source  :', {f"{ip} ({IP_TO_NODE.get(ip,'?')})": c for ip, c in sorted(by_src.items(), key=lambda x: -x[1])})
    if not ALERTS:
        print('\nNo alerts yet — fast.log tail:')
        print(sh(SENSOR, 'sudo tail -n 20 /var/log/suricata/fast.log 2>/dev/null').stdout or '(none yet)')


## 5. Visualize — categories, severity, top talkers, attack graph, timeline

In [ ]:
# Visualizations. Plotting libs are installed into the kernel if missing
# (subprocess, not %magic, so the cell stays valid Python everywhere).
import sys, subprocess
def _ensure(mod, pip=None):
    try:
        __import__(mod)
    except Exception:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip or mod])
_ensure('matplotlib'); _ensure('networkx')
import matplotlib.pyplot as plt
import networkx as nx
from datetime import datetime

LINKS = [["edge", "edge2"], ["gw", "mon"]]
def lbl(ip):
    return f"{IP_TO_NODE.get(ip, ip)}"

# Figure 1: topology + alerts by signature
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
G = nx.Graph(); G.add_nodes_from(NODES); G.add_edges_from(LINKS)
col = ['#e25241' if n == 'edge' else '#008e7a' if n == 'edge2' else '#5798bc' for n in G.nodes()]
nx.draw(G, ax=ax[0], with_labels=True, node_color=col, node_size=1900,
        font_color='white', font_weight='bold', edge_color='#888', width=2)
ax[0].set_title('Topology (red=attacker, green=benign)')
if counts:
    top = sorted(counts, key=lambda s: -counts[s])[:8][::-1]
    ax[1].barh([s.replace('EDGEIDS ', '')[:34] for s in top], [counts[s] for s in top], color='#ff8542')
    ax[1].set_title('Alerts by signature'); ax[1].set_xlabel('count')
else:
    ax[1].text(.5, .5, 'no alerts yet', ha='center'); ax[1].set_axis_off()
plt.tight_layout(); plt.show()

# Figure 2: alert categories + severity
if by_cat or by_sev:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    cats = sorted(by_cat, key=lambda c: -by_cat[c])
    ax[0].bar([c[:18] for c in cats], [by_cat[c] for c in cats], color='#1f6a8c')
    ax[0].set_title('Alerts by category'); ax[0].tick_params(axis='x', rotation=30)
    sevmap = {1: 'high', 2: 'medium', 3: 'low'}
    sevs = sorted(by_sev)
    ax[1].pie([by_sev[s] for s in sevs], labels=[sevmap.get(s, str(s)) for s in sevs],
              autopct='%1.0f%%', colors=['#e25241', '#ff8542', '#5798bc'])
    ax[1].set_title('Alert severity')
    plt.tight_layout(); plt.show()

# Figure 3: top talkers (benign vs compromised) + attack graph
if by_src:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    srcs = sorted(by_src, key=lambda s: -by_src[s])
    bar_c = ['#e25241' if lbl(s) == 'edge' else '#008e7a' for s in srcs]
    ax[0].bar([f'{lbl(s)}\n{s}' for s in srcs], [by_src[s] for s in srcs], color=bar_c)
    ax[0].set_title('Top talkers — alerts by source (benign vs compromised)')
    ax[0].set_ylabel('alerts')
    DG = nx.DiGraph()
    pair = {}
    for a in ALERTS:
        pair[(lbl(a['src']), lbl(a['dest']))] = pair.get((lbl(a['src']), lbl(a['dest'])), 0) + 1
    for (s, d), w in pair.items():
        DG.add_edge(s, d, weight=w)
    if DG.number_of_edges():
        pos = nx.spring_layout(DG, seed=1)
        ws = [DG[u][v]['weight'] for u, v in DG.edges()]
        mx = max(ws)
        nx.draw(DG, pos, ax=ax[1], with_labels=True, node_color='#5798bc',
                node_size=1500, font_color='white', font_weight='bold',
                edge_color='#e25241', width=[1 + 4 * w / mx for w in ws],
                arrowsize=18)
        ax[1].set_title('Attack graph (src → dest, width = #alerts)')
    else:
        ax[1].text(.5, .5, 'no alert edges', ha='center'); ax[1].set_axis_off()
    plt.tight_layout(); plt.show()

# Figure 4: alerts over time (10s bins)
def _t(s):
    try:
        return datetime.fromisoformat(s.replace('Z', '+00:00'))
    except Exception:
        return None
times = sorted(t for t in (_t(a['ts']) for a in ALERTS) if t)
if times:
    t0 = times[0]; bins = {}
    for t in times:
        k = int((t - t0).total_seconds() // 10) * 10
        bins[k] = bins.get(k, 0) + 1
    xs = sorted(bins)
    plt.figure(figsize=(11, 3))
    plt.bar([f'{x}s' for x in xs], [bins[x] for x in xs], color='#008e7a')
    plt.title('Alerts over time (10s bins, since first alert)')
    plt.xlabel('seconds'); plt.ylabel('alerts'); plt.tight_layout(); plt.show()


## 6. Respond (IPS) — auto-block the attacker, keep the benign device

In [ ]:
# Closed-loop IPS: identify the top attacker from the alerts and DROP it on
# the gateway with nftables, then prove the benign device still works.
def web(node, gw_ip):
    if not (node and gw_ip):
        return 'n/a'
    r = sh(node, f"curl -s -m 5 -o /dev/null -w '%{{http_code}}' http://{gw_ip}/ 2>/dev/null || echo 000")
    return (r.stdout or '000').strip()

if not GATEWAY:
    print('No gateway node identified.')
elif not by_src:
    print('No alerts collected yet — run the Analyze cell first.')
else:
    attacker = max(by_src, key=by_src.get)
    atk_node = IP_TO_NODE.get(attacker, '?')
    gw_ip = next((a.split('/')[0] for a in NODE_ADDRS.get(GATEWAY, []) if a.startswith('10.0.1.')), None)
    benign = [n for n in NODES if n not in (GATEWAY, atk_node)
              and any(a.startswith('10.0.1.') for a in NODE_ADDRS.get(n, []))]
    bnode = benign[0] if benign else None
    print(f'Top attacker: {attacker} ({atk_node}) — {by_src[attacker]} alerts. Enforcing on {GATEWAY}.')
    print(f'BEFORE block:  attacker({atk_node}) http={web(atk_node, gw_ip)}   benign({bnode}) http={web(bnode, gw_ip)}')
    blk = sh(GATEWAY, f'sudo nft add rule inet filter input ip saddr {attacker} drop; '
                      f'sudo nft add rule inet filter forward ip saddr {attacker} drop; echo OK')
    print('IPS rule:', (blk.stdout or blk.stderr).strip())
    time.sleep(3)
    print(f'AFTER block:   attacker({atk_node}) http={web(atk_node, gw_ip)}   benign({bnode}) http={web(bnode, gw_ip)}')
    print(f'\n=> The attacker is cut off (http 000) while {bnode} keeps working (http 200).')
    print(f'Undo:  ssh {GATEWAY} "sudo nft flush chain inet filter input; sudo nft flush chain inet filter forward"')
